# 11c Calibrated safety stock

In 11b the safety stock comes from the textbook formula, z times the error spread times the square root of the lead
time. That formula assumes forecast errors are unbiased, normal and independent from day to day. 07b and 11a showed
none of that holds. Here we size safety stock from the errors themselves instead, the same way 11a builds its bands:
pool each category's past 7-day error sums (scaled per product), take the quantile for the target service level, and
scale it back to the product. Then we rerun the simulation and compare.

## Load

In [1]:
import sys, warnings
sys.path.append("../src")
from config import BEST_NAME
BEST_SHORT = BEST_NAME.split(" (")[0]      # the best time-series model from 07h, stored as 'best'
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import ROOT, RAW, INTERIM, PROCESSED, TABLES, FIGS, FIG_STYLE
from break_detection import weekly_ca_prices
from metrics import paired_test
from pipeline import price_daily_frame, simulate_inventory, summarise_inventory, intermittency

SEED = 42
plt.rcParams.update(FIG_STYLE)
written = []
fc = pd.read_parquet(PROCESSED / "backtest_forecasts.parquet")
err = pd.read_csv(PROCESSED / "backtest_errors.csv", parse_dates=["origin"])
scales = err[["item_id", "origin", "mase_scale"]].drop_duplicates()
ORIGINS = sorted(fc["origin"].unique())
EVAL = ORIGINS[1:]
fc["category"] = fc["item_id"].str.split("_").str[0]

POL = [("best", "static"), ("best", "adaptive"), ("seasonal_naive", "static")]
f = fc[fc.set_index(["model", "variant"]).index.isin(POL)].merge(scales, on=["item_id", "origin"])
f["e"] = f["actual"] - f["forecast"]
f["r"] = f["e"] / f["mase_scale"]          # errors in units of each product's own MASE scale, so products can be pooled
print(len(ORIGINS), "origins;", f"{len(f):,} forecast days for", POL)

7 origins; 426,888 forecast days for [('best', 'static'), ('best', 'adaptive'), ('seasonal_naive', 'static')]


## The textbook version, as a check against 11b

In [2]:
long_df = pd.concat([pd.read_csv(INTERIM / f"{c}_candidates.csv", parse_dates=["date"])
                     for c in ["foods", "hobbies", "household"]], ignore_index=True)
calendar = pd.read_csv(RAW / "calendar.csv", parse_dates=["date"])
prices = pd.read_csv(RAW / "sell_prices.csv", dtype={"store_id": "category"})
price_wk = weekly_ca_prices(prices, calendar, long_df["item_id"].unique())
del prices
days = pd.date_range(EVAL[0] + pd.Timedelta(days=1), periods=28 * len(EVAL))
price_daily = price_daily_frame(price_wk, days)
inter = intermittency(long_df, ORIGINS[0])
L, SLS = 7, [0.90, 0.95, 0.99]

normal = simulate_inventory(fc, ORIGINS, price_daily, POL, SLS, L, inter)
ns = summarise_inventory(normal)
ref = pd.read_csv(TABLES / "inventory_policy_summary.csv")
chk = ns.merge(ref, on=["policy", "service_level"], suffixes=("", "_11b"))
print("largest difference from 11b: total cost", np.abs(chk["total_cost"] - chk["total_cost_11b"]).max().round(6),
      "| fill rate", np.abs(chk["fill_rate"] - chk["fill_rate_11b"]).max().round(6), "(11b rounds to 3 decimals)")

largest difference from 11b: total cost 0.00048 | fill rate 0.000472 (11b rounds to 3 decimals)


## Safety stock from the errors

In [3]:
sums = []
for (item, origin, model, variant, cat), g in f.groupby(["item_id", "origin", "model", "variant", "category"]):
    s = np.convolve(g.sort_values("date")["e"].to_numpy(), np.ones(L), mode="valid") / g["mase_scale"].iloc[0]
    sums.append(pd.DataFrame({"origin": origin, "model": model, "variant": variant, "category": cat, "r7": s}))
sums = pd.concat(sums, ignore_index=True)
scale_map = scales.set_index(["item_id", "origin"])["mase_scale"]
cat_map = long_df.groupby("item_id")["item_id"].first().str.split("_").str[0]
qcache = {}

def conformal_ss(item, model, variant, origin, lead_time, sl, past_errors):
    """Pooled, scaled 7-day error quantile from earlier origins, scaled back to the product."""
    cat = cat_map[item]
    key = (cat, model, variant, origin, sl)
    if key not in qcache:
        pool = sums[(sums["category"] == cat) & (sums["model"] == model) & (sums["variant"] == variant)
                    & (sums["origin"] < origin)]["r7"].to_numpy()
        n = len(pool)
        qcache[key] = np.quantile(pool, min(1.0, np.ceil((n + 1) * sl) / n), method="higher")
    return max(0.0, scale_map[(item, origin)] * qcache[key])

conf = simulate_inventory(fc, ORIGINS, price_daily, POL, SLS, L, inter, ss_fn=conformal_ss)
both = pd.concat([normal.assign(method="normal formula"), conf.assign(method="conformal")], ignore_index=True)
both.to_csv(TABLES / "inventory_calibrated_items.csv", index=False)
inv = summarise_inventory(both, by=("method", "policy", "service_level"))
inv.to_csv(TABLES / "inventory_calibrated_vs_normal.csv", index=False)
written += [TABLES / "inventory_calibrated_items.csv", TABLES / "inventory_calibrated_vs_normal.csv"]
inv[["method", "policy", "service_level", "fill_rate", "cycle_service_level", "cycles", "stockout_days",
     "avg_inventory_value", "holding_cost", "stockout_cost", "total_cost"]].round(4)

,method,policy,service_level,fill_rate,cycle_service_level,cycles,stockout_days,avg_inventory_value,holding_cost,stockout_cost,total_cost
0,conformal,best_adaptive,0.90,0.9744,0.8252,15221,2989,176440.9558,20302.7949,54727.8807,75030.6756
1,conformal,best_adaptive,0.95,0.9829,0.8817,15420,1876,199925.1798,23005.0892,37486.6209,60491.7101
2,conformal,best_adaptive,0.99,0.9939,0.9489,15784,597,265890.8024,30595.6540,15654.8668,46250.5208
3,conformal,best_static,0.90,0.9747,0.8238,15283,2991,175724.2621,20220.3260,54650.5628,74870.8888
4,conformal,best_static,0.95,0.9829,0.8816,15488,1859,199093.1430,22909.3480,37757.3270,60666.6750
5,conformal,best_static,0.99,0.9938,0.9490,15819,587,266478.8123,30663.3154,15363.8428,46027.1582
6,conformal,seasonal_naive_static,0.90,0.9726,0.8027,15166,3247,181120.9986,20841.3204,58673.6328,79514.9531
7,conformal,seasonal_naive_static,0.95,0.9820,0.8683,15435,1990,205453.9167,23641.2726,39204.9549,62846.2275
8,conformal,seasonal_naive_static,0.99,0.9937,0.9420,15817,649,274745.9671,31614.6044,15570.7421,47185.3465
9,normal formula,best_adaptive,0.90,0.9645,0.7722,14764,4398,167039.5859,19220.9935,72108.8100,91329.8035


In [4]:
rows = []
for (model, variant), _ in both.groupby(["model", "variant"]):
    for sl in SLS:
        p = both[(both["model"] == model) & (both["variant"] == variant) & (both["service_level"] == sl)]
        w = p.pivot(index="item_id", columns="method", values="total_cost")
        rows.append({"policy": f"{model}_{variant}", "service_level": sl,
                     **paired_test(w["conformal"] - w["normal formula"], seed=SEED)})
cmp_ = pd.DataFrame(rows)
cmp_.to_csv(TABLES / "inventory_calibrated_paired.csv", index=False)
written.append(TABLES / "inventory_calibrated_paired.csv")
print("per-product total cost, conformal minus textbook:")
cmp_[["policy", "service_level", "n", "median_diff", "ci_low", "ci_high", "share_improved", "p_value"]].round(4)

per-product total cost, conformal minus textbook:


,policy,service_level,n,median_diff,ci_low,ci_high,share_improved,p_value
0,best_adaptive,0.90,726,-8.6363,-10.0207,-7.1514,0.7562,0.0000
1,best_adaptive,0.95,726,-8.2080,-9.3207,-6.0143,0.7231,0.0000
2,best_adaptive,0.99,726,1.2230,0.4165,1.8804,0.4559,0.0001
3,best_static,0.90,726,-7.8602,-9.7879,-6.3946,0.7479,0.0000
4,best_static,0.95,726,-6.6876,-8.6179,-5.1524,0.7052,0.0000
5,best_static,0.99,726,1.1220,0.4424,1.6701,0.4518,0.0004
6,seasonal_naive_static,0.90,726,-0.7355,-1.2329,-0.2748,0.5579,0.0000
7,seasonal_naive_static,0.95,726,-1.1203,-1.9078,-0.5012,0.5909,0.0000
8,seasonal_naive_static,0.99,726,1.8271,1.3457,2.5610,0.3898,0.7372


## What we found

The textbook formula promises more than it delivers. With the combination's forecasts and a 95% target, only 81.5% of
order cycles got through without a stockout; with a 99% target, 88.1%.

Safety stock sized from the real errors gets much closer, though not all the way: 88.2% of cycles at the 95% target
and 94.9% at 99%. Fill rate goes from 97.1% to 98.3% at 95%. It holds more stock ($199,093 of average inventory
against $176,693), but loses so many fewer sales that total cost falls from $80,250 to $60,667, which is 24.4% lower,
and it is cheaper for 71% of products. At the 99% target the total is lower too ($46,027 against $68,019), but the
typical product pays slightly more (median +$1.12, 95% CI +$0.44 to +$1.67): there the saving comes from a minority
of products.

With this safety stock the better forecast pays after all: the combination costs $60,667 against $62,846 for
seasonal naive at 95%, 3.5% less, the reverse of 11b. A better forecast lowers cost only when the safety stock is
sized from its real errors.

These costs use our assumed prices for holding stock and losing a sale; the fill rates and cycle counts come
straight from the simulation.

With SARIMA the cut was 24.1%; with LightGBM, in the earlier version of the project, calibrated safety stock cut
total cost by 31.1% and reached 91.7% of cycles at the 95% target.

## What comes next

`11d_inventory_other_seasons` checks whether the stock results hold in two other half-years.

In [5]:
# files this notebook wrote
for p in written:
    print(p.relative_to(ROOT), f"{p.stat().st_size / 1e3:,.0f} KB")

outputs\tables\inventory_calibrated_items.csv 3,178 KB
outputs\tables\inventory_calibrated_vs_normal.csv 4 KB
outputs\tables\inventory_calibrated_paired.csv 2 KB
